In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
working_directory = "/Users/kemalinecik/git_nosync/sctram"

In [3]:
import sys
sys.path.append(working_directory)

import logging
import os
import numpy as np
import pandas as pd
import networkx as nx
import scanpy as sc
import anndata as ad
from sctram.generate.real import sc_norman_sciplex_cpa

In [4]:
from sctram.api._lower_level import TrajectoryEvaluationAPI
from sctram.input import read_dict

2025-02-26 17:00:24.681 | INFO     | sctram.api._defaults_read:load_default_metrics:23 - Loaded default metrics from /Users/kemalinecik/git_nosync/sctram/sctram/api/_defaults.yaml
2025-02-26 17:00:24.682 | INFO     | sctram.api._defaults_read:load_default_metrics:79 - Default metrics YAML structure validated successfully.


In [5]:
dataset_dir = "/Users/kemalinecik/git_nosync/sctram/__temp__/data"
adata_norman = sc_norman_sciplex_cpa(dataset_dir=dataset_dir)
adata_bms = adata_norman[["bms" in i.lower() or "vehicle" in i.lower() for i in adata_norman.obs["drug"]]]


2025-02-26 17:00:24.741 | WARNING  | sctram.generate.real._download:download_dataset:105 - File PosixPath('/Users/kemalinecik/git_nosync/sctram/__temp__/data/norman_sciplex_cpa.h5ad') already exists. Skipping download.


In [6]:
ground_truth_trajectories = {
    "trajectory_1": [
        ('Vehicle_1.0', 'BMS_0.001'),
        ('BMS_0.001', 'BMS_0.005'),
        ('BMS_0.005', 'BMS_0.01'),
        ('BMS_0.01', 'BMS_0.05'),
        ('BMS_0.05', 'BMS_0.1'),
        ('BMS_0.1', 'BMS_0.5'),
        ('BMS_0.5', 'BMS_1.0'),
    ],
}
input_trajectories_all = read_dict(ground_truth_trajectories)
input_trajectories = input_trajectories_all.get_trajectory("trajectory_1", include_additional_nodes=False)

In [7]:
adata = ad.AnnData(X=adata_bms.obsm["tardis"][:,:16].copy(), obs=adata_bms.obs.copy())
api = TrajectoryEvaluationAPI(
    adata=adata,
    input_trajectories=input_trajectories,
    labels_obs="drug_dose_name",
    root_label="Vehicle_1.0",
    logger_level="DEBUG"
)
api.evaluate_with_defaults()

2025-02-26 17:00:24.993 | INFO     | sctram.api._lower_level:evaluate_adjacency:154 - Starting adjacency evaluation.
2025-02-26 17:00:24.994 | INFO     | sctram.api._lower_level:_get_inference_method:79 - Running pseudotime inference with method 'PAGAInference'
2025-02-26 17:00:24.994 | INFO     | sctram.api._lower_level:_get_evaluate_method:72 - Running pseudotime evaluation with method 'AdjacencyMatrixEvaluation'
2025-02-26 17:00:24.999 | DEBUG    | sctram.infer._base:_initialize_from_adata_without_neighbors:161 - Initializing from AnnData without precomputed neighbors.
2025-02-26 17:00:25.001 | DEBUG    | sctram.infer._base:_add_labels_to_adata:220 - Adding provided labels to AnnData object.
2025-02-26 17:00:25.002 | INFO     | sctram.infer._base:_initialize_from_adata_without_neighbors:168 - No precomputed neighbors found in AnnData.
2025-02-26 17:00:25.002 | DEBUG    | sctram.infer._base:_initialize_from_adata_without_neighbors:172 - AnnData initialized successfully from AnnData w

In [8]:
adata_scvi = ad.AnnData(X=adata_bms.obsm["scvi"].copy(), obs=adata_bms.obs.copy())
api_scvi = TrajectoryEvaluationAPI(
    adata=adata_scvi,
    input_trajectories=input_trajectories,
    labels_obs="drug_dose_name",
    root_label="Vehicle_1.0",
    logger_level="WARNING"
)
api_scvi.evaluate_with_defaults()

2025-02-26 17:00:35.548 | WARNING  | sctram.utils._utils:sget:34 - Default value 0.5 used for missing key 'alpha'.
2025-02-26 17:00:35.549 | WARNING  | sctram.utils._utils:sget:34 - Default value 100 used for missing key 'n_steps'.
2025-02-26 17:00:35.549 | WARNING  | sctram.utils._utils:sget:34 - Default value 1e-06 used for missing key 'tol'.
2025-02-26 17:00:37.965 | WARNING  | sctram.evaluate._metrics._trajectory_cardinality_validation:trajectory_cardinality_validation:75 - `trajectory_cardinality_validation` is not tested.


In [9]:
adata_pca = ad.AnnData(X=adata_bms.obsm["X_pca"].copy(), obs=adata_bms.obs.copy())
api_pca = TrajectoryEvaluationAPI(
    adata=adata_pca,
    input_trajectories=input_trajectories,
    labels_obs="drug_dose_name",
    root_label="Vehicle_1.0",
    logger_level="INFO"
)
api_pca.evaluate_with_defaults()

2025-02-26 17:00:42.363 | INFO     | sctram.api._lower_level:evaluate_adjacency:154 - Starting adjacency evaluation.
2025-02-26 17:00:42.363 | INFO     | sctram.api._lower_level:_get_inference_method:79 - Running pseudotime inference with method 'PAGAInference'
2025-02-26 17:00:42.363 | INFO     | sctram.api._lower_level:_get_evaluate_method:72 - Running pseudotime evaluation with method 'AdjacencyMatrixEvaluation'
2025-02-26 17:00:42.369 | INFO     | sctram.infer._base:_initialize_from_adata_without_neighbors:168 - No precomputed neighbors found in AnnData.
2025-02-26 17:00:42.370 | INFO     | sctram.infer._base:calculate:496 - Starting PAGAInference trajectory inference.
2025-02-26 17:00:42.370 | INFO     | sctram.infer._base:calculate:499 - Performing trajectory inference.
2025-02-26 17:00:42.370 | INFO     | sctram.infer._base:_needs_neighbors:514 - Computing neighbors.
2025-02-26 17:00:42.370 | INFO     | sctram.utils._loguru_scanpy_capture:emit:51 - computing neighbors
2025-02-26

In [10]:
df = api.get_all_results()
df_scvi = api_scvi.get_all_results()
df_pca = api_pca.get_all_results()
df["score_tardis"] = df["score"]
df["score_scvi"] = df_scvi["score"]
df["score_pca"] = df_pca["score"]
del df["score"]
df

,path,metric,score_tardis,score_scvi,score_pca
0,adjacency,frobenius,2.533942,3.758729,3.991882
1,adjacency,l1_norm,12.866654,22.495203,24.590187
2,adjacency,accuracy,0.906250,0.531250,0.468750
3,adjacency,graph_edit_distance,3.000000,15.000000,17.000000
4,adjacency,spectral_distance,1.837844,2.729414,3.039379
5,adjacency,jaccard_similarity,0.625000,0.250000,0.260870
6,adjacency,hamming_distance,6.000000,30.000000,34.000000
7,adjacency,precision,0.833333,0.277778,0.272727
8,adjacency,recall,0.714286,0.714286,0.857143
9,adjacency,f1_score,0.769231,0.400000,0.413793
